# NPTEL Dataset Cleaning

This notebook performs data cleaning on the NPTEL.csv dataset. The cleaning steps include:
1. Reading and examining the data
2. Fixing column names and structure
3. Handling missing values
4. Standardizing date formats
5. Cleaning text fields and encoding issues
6. Converting data types
7. Organizing and consolidating columns
8. Saving the cleaned dataset

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime
import re

# Read the dataset
df = pd.read_csv('../Data/raw/NPTEL.csv')

In [2]:
# Display basic information about the dataset
print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDataset Info:")
df.info()

Dataset Shape: (4136, 756)

Columns: ['course_name', 'abstract', 'instructor', 'course_timeline.course_duration.start_date', 'course_timeline.course_duration.end_date', 'course_timeline.enrollment.start_date', 'course_timeline.enrollment.end_date', 'course_timeline.exam_registration.start_date', 'course_timeline.exam_registration.end_date', 'course_timeline.exam_date', 'metrics', 'course_url', 'enrolled_role.student', 'enrolled_role.faculty', 'enrolled_role.others', 'enrolled_role.employed', 'enrolled_age_groups.13-20', 'enrolled_age_groups.20-30', 'enrolled_age_groups.30-40', 'enrolled_age_groups.40-50', 'enrolled_age_groups.50-60', 'enrolled_age_groups.60-70', 'enrolled_age_groups.70-80', 'enrolled_age_groups.80-90', 'enrolled_country.United States', 'enrolled_country.New Zealand', 'enrolled_country.Saint Lucia', 'enrolled_country.Australia', 'enrolled_country.Belgium', 'enrolled_state.maharashtra', 'enrolled_state.telengana', 'enrolled_state.madhyapradesh', 'enrolled_state.andhrapra

In [3]:
# Display first few rows of the dataset
df.head()

,course_name,abstract,instructor,course_timeline.course_duration.start_date,course_timeline.course_duration.end_date,course_timeline.enrollment.start_date,course_timeline.enrollment.end_date,course_timeline.exam_registration.start_date,course_timeline.exam_registration.end_date,course_timeline.exam_date,...,weekly_avg_score.4.4.obj,weekly_submission_count.4.4.obj,weekly_avg_score.8.0.complete,weekly_submission_count.8.0.complete,registered_state.timphu,registered_state.mahakali,weekly_avg_score.12.4.obj,weekly_submission_count.12.4.obj,registered_state.london,registered_state.california
0,Introduction to Soft Matter,Introductory course on soft matter/complex flu...,Prof. Aloke Kumar,Feb-22,Apr-22,14-Nov-21,21-Feb-22,13-Dec-21,18-Mar-22,23-Apr-22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Powder Metallurgy,Powder Metallurgy is a very useful manufacturi...,Prof. Ranjit Bauri,Jul-21,Oct-21,20-May-21,02-Aug-21,17-Jun-21,17-Sep-21,24-Oct-21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Optical Sensors,This course provides detailed insight of the f...,Prof. Sachin Kumar Srivastava,Jan-22,Feb-22,14-Nov-21,31-Jan-22,13-Dec-21,18-Feb-22,27-Mar-22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Financial Mathematics,The course on Financial Mathematics focuses on...,Prof. Pradeep Kumar Jha,Jan-21,Apr-21,18-Nov-20,25-Jan-21,15-Jan-21,12-Mar-21,25-Apr-21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Biostatistics and design of experiments,Biostatistics is the application of statistics...,Prof. Mukesh Doble,Jul-17,Sep-17,17-May-17,24-Jul-17,02-Aug-17,23-Aug-17,24-Sep-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Check for duplicate entries
print("Number of duplicate rows:", df.duplicated().sum())

# Remove duplicates if any exist
df = df.drop_duplicates().reset_index(drop=True)

# Check missing values
print("\nMissing values in each column:")
print(df.isnull().sum())

# Standardize column names
# Convert to lowercase and replace spaces with underscores
df.columns = df.columns.str.lower().str.replace(' ', '_')

Number of duplicate rows: 0

Missing values in each column:
course_name                                      0
abstract                                        22
instructor                                      42
course_timeline.course_duration.start_date       0
course_timeline.course_duration.end_date         0
                                              ... 
registered_state.mahakali                     4135
weekly_avg_score.12.4.obj                     4135
weekly_submission_count.12.4.obj              4135
registered_state.london                       4135
registered_state.california                   4135
Length: 756, dtype: int64


In [8]:
# Flatten nested column names
df.columns = df.columns.str.replace('.', '_')

# Select essential columns
essential_columns = [
    'course_name', 'abstract', 'instructor',
    'course_timeline_course_duration_start_date',
    'course_timeline_course_duration_end_date'
]

# Keep only essential columns
df = df[essential_columns].copy()  # Create a copy to avoid SettingWithCopyWarning

# Clean text fields
df['course_name'] = df['course_name'].str.strip()
df['abstract'] = df['abstract'].fillna('').str.strip()
df['instructor'] = df['instructor'].fillna('Unknown').str.strip()

# Function to convert date format 'MMM-YY' to datetime
def convert_date(date_str):
    if pd.isna(date_str):
        return None
    try:
        # Convert 'MMM-YY' to datetime, assuming dates are from 2000s
        date = datetime.strptime(date_str, '%b-%y')
        # If the year is after current year, assume it's from 1900s
        if date.year > datetime.now().year:
            date = date.replace(year=date.year - 100)
        return date
    except:
        return None

# Convert dates
df['course_timeline_course_duration_start_date'] = df['course_timeline_course_duration_start_date'].apply(convert_date)
df['course_timeline_course_duration_end_date'] = df['course_timeline_course_duration_end_date'].apply(convert_date)

# Display the cleaned dataset info
print("Cleaned Dataset Info:")
df.info()
print("\nSample of cleaned data:")
display(df.head())

Cleaned Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4136 entries, 0 to 4135
Data columns (total 5 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   course_name                                 4136 non-null   object        
 1   abstract                                    4136 non-null   object        
 2   instructor                                  4136 non-null   object        
 3   course_timeline_course_duration_start_date  4135 non-null   datetime64[ns]
 4   course_timeline_course_duration_end_date    4136 non-null   datetime64[ns]
dtypes: datetime64[ns](2), object(3)
memory usage: 161.7+ KB

Sample of cleaned data:


,course_name,abstract,instructor,course_timeline_course_duration_start_date,course_timeline_course_duration_end_date
0,Introduction to Soft Matter,Introductory course on soft matter/complex flu...,Prof. Aloke Kumar,2022-02-01,2022-04-01
1,Powder Metallurgy,Powder Metallurgy is a very useful manufacturi...,Prof. Ranjit Bauri,2021-07-01,2021-10-01
2,Optical Sensors,This course provides detailed insight of the f...,Prof. Sachin Kumar Srivastava,2022-01-01,2022-02-01
3,Financial Mathematics,The course on Financial Mathematics focuses on...,Prof. Pradeep Kumar Jha,2021-01-01,2021-04-01
4,Biostatistics and design of experiments,Biostatistics is the application of statistics...,Prof. Mukesh Doble,2017-07-01,2017-09-01


In [9]:
# Save the cleaned dataset
output_path = '../Data/processed/nptel_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to: {output_path}")

# Display some basic statistics about the cleaned dataset
print("\nDataset Statistics:")
print(f"Total number of courses: {len(df)}")
print(f"Number of unique instructors: {df['instructor'].nunique()}")
print(f"\nDate range of courses:")
print(f"Earliest start date: {df['course_timeline_course_duration_start_date'].min()}")
print(f"Latest end date: {df['course_timeline_course_duration_end_date'].max()}")

Cleaned dataset saved to: ../Data/processed/nptel_cleaned.csv

Dataset Statistics:
Total number of courses: 4136
Number of unique instructors: 1358

Date range of courses:
Earliest start date: 2014-03-01 00:00:00
Latest end date: 2022-04-01 00:00:00
